In [ ]:
import os
import glob
import cv2
import numpy as np
import pandas as pd
from dataset.data_loader import BaseLoader

class CustomDirLoader(BaseLoader):
    """
    Custom loader for reading `.bmp` image sequences and `.txt` pulse wave labels.
    """
    def __init__(self, dataset_name, raw_data_path, label_data_path, config_data, device=None):
        self.label_data_path = label_data_path
        super().__init__(dataset_name, raw_data_path, config_data, device)

    def get_raw_data(self, raw_data_path):
        """
        Locates directories, ensuring both video (.bmp) and labels (.txt) exist.
        """
        data_dirs = []
        participant_folders = [f.path for f in os.scandir(raw_data_path) if f.is_dir()]
        
        for p_folder in participant_folders:
            participant_id = os.path.basename(p_folder)
            label_path = os.path.join(self.label_data_path, f"{participant_id}.txt")
            
            # Condition 1: Must have a pulse file
            if not os.path.exists(label_path):
                continue
            
            subfolders = [f.path for f in os.scandir(p_folder) if f.is_dir()]
            img_folder = subfolders[0] if subfolders else p_folder
            
            # Condition 2: Must have video files
            if not glob.glob(os.path.join(img_folder, '*.bmp')):
                continue
                
            data_dirs.append({
                'index': participant_id,
                'path': img_folder
            })
            
        return data_dirs

    def split_raw_data(self, data_dirs, begin, end):
        total = len(data_dirs)
        start_idx = int(round(begin * total))
        end_idx = int(round(end * total))
        return data_dirs[start_idx:end_idx]

    def read_bmp_video(self, image_dir):
        image_paths = sorted(glob.glob(os.path.join(image_dir, '*.bmp')))
        
        frames = []
        for img_path in image_paths:
            img = cv2.imread(img_path)
            if img is not None:
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                frames.append(img)
                
        return np.asarray(frames)

    def read_pulse_wave(self, participant_id):
        """Reads the tab-separated .txt pulse wave label, keeping only the first column."""
        label_path = os.path.join(self.label_data_path, f"{participant_id}.txt")
        pulse_wave = np.loadtxt(label_path, delimiter='\t', usecols=0) 
        return pulse_wave

    def preprocess_dataset_subprocess(self, data_dirs, config_preprocess, i, file_list_dict):
        """
        Handles reading, resampling using BaseLoader.resample_ppg, truncating, and passing to the BaseLoader core.
        """
        data_info = data_dirs[i]
        participant_id = data_info['index']
        img_dir = data_info['path']

        # 1. Read Raw Data
        frames = self.read_bmp_video(img_dir)
        bvps = self.read_pulse_wave(participant_id)

        # 2. Determine original signal sampling rate (500Hz or 1000Hz)
        # We know video is exactly 30 fps. Calculate expected duration.
        video_duration_sec = len(frames) / 30.0
        
        # Check which sampling rate fits the signal length better
        diff_500 = abs(len(bvps) - (video_duration_sec * 500))
        diff_1000 = abs(len(bvps) - (video_duration_sec * 1000))
        original_fs = 500 if diff_500 < diff_1000 else 1000

        # 3. Resample the signal down to match the 30Hz video framerate
        # target length = (current length * target rate) / original rate
        target_signal_length = int(len(bvps) * (30.0 / original_fs))
        
        # Using the base loader's native static method for downsampling
        bvps_resampled = BaseLoader.resample_ppg(bvps, target_signal_length)

        # 4. Truncate to the shorter of the two lengths to ensure perfect alignment
        min_length = min(len(frames), len(bvps_resampled))
        frames = frames[:min_length]
        bvps_resampled = bvps_resampled[:min_length]

        # 5. Pass to BaseLoader's core preprocessing (face crop, resize, chunking)
        frames_clips, bvps_clips = self.preprocess(frames, bvps_resampled, config_preprocess)

        # 6. Save chunked data
        input_paths, label_paths = self.save_multi_process(frames_clips, bvps_clips, participant_id)
        file_list_dict[i] = input_paths

In [ ]:
import os

class ConfigMock:
    def __init__(self, dictionary):
        for key, value in dictionary.items():
            if isinstance(value, dict):
                setattr(self, key, ConfigMock(value))
            else:
                setattr(self, key, value)

if __name__ == '__main__':
    RAW_DATA_DIR = 'G:/image/relax'
    LABEL_DATA_DIR = 'G:/pulse_data/relax'
    CACHE_DIR = 'G:/data/preprocessed_cache_72_150'
    FILE_LIST = 'G:/data/file_list.csv'

    os.makedirs(CACHE_DIR, exist_ok=True)

    config_dict = {
        'CACHED_PATH': CACHE_DIR,
        'FILE_LIST_PATH': FILE_LIST,
        'DATA_FORMAT': 'NDCHW',
        'DO_PREPROCESS': True, 
        'BEGIN': 0.0,          
        'END': 1.0,            
        'PREPROCESS': {
            'DATA_TYPE': ['Standardized'],    
            'LABEL_TYPE': 'Raw',        
            'DO_CHUNK': True,
            'CHUNK_LENGTH': 150,        
            'RESIZE': {
                'H': 72,
                'W': 72
            },
            'CROP_FACE': {
                'DO_CROP_FACE': True,
                'BACKEND': 'Y5F',        
                'USE_LARGE_FACE_BOX': True,
                'LARGE_BOX_COEF': 1.5,
                'DETECTION': {
                    'DO_DYNAMIC_DETECTION': False,
                    'DYNAMIC_DETECTION_FREQUENCY': 30,
                    'USE_MEDIAN_FACE_BOX': False
                }
            }
        }
    }
    
    config_data = ConfigMock(config_dict)

    print("Starting Preprocessing Pipeline...")
    
    loader = CustomDirLoader(
        dataset_name="BMP_Sequence_Dataset",
        raw_data_path=RAW_DATA_DIR,
        label_data_path=LABEL_DATA_DIR,
        config_data=config_data,
        device='cpu' 
    )
    
    print("\nPreprocessing Complete.")